# 624 Hektor

## Initialize

In [201]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
from astroquery.jplhorizons import Horizons

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

In [202]:
DAYS_PER_JULIAN_YEAR = 365.25

OBJECT = {
    'SUN': {
        'id': '10',
        'name': 'Sun',
        'color': 'darkorange'
    },
    'JUPITER': {
        'id': '5',
        'name': 'Jupiter',
        'color': 'chocolate'
    },
    'HEKTOR': {
        'id': 'Hektor',
        'name': '624 Hektor',
        'color': 'lightseagreen'
    }
}

EPOCH_START_JD = 2461143.5

In [203]:
# Mercury6 data
col = ['time', 'long', 'x', 'y', 'z', 'vx', 'vy', 'vz']

vec_j = pd.read_csv('JUPITER.aei', sep='\s+', skiprows=4, names=col)
vec_h = pd.read_csv('HEKTOR.aei', sep='\s+', skiprows=4, names=col)

In [ ]:
# Horizons Data
duration_years = 10
stop_jd = EPOCH_START_JD + (duration_years * DAYS_PER_JULIAN_YEAR)

start_iso = Time(EPOCH_START_JD, format='jd').to_value('iso', subfmt='date')
stop_iso = Time(stop_jd, format='jd').to_value('iso', subfmt='date')

epochs = {
    'start': start_iso,
    'stop': stop_iso,
    'step': '60d'
}

obj_j_hor = Horizons(id=OBJECT['JUPITER']['id'], location='@sun', epochs=epochs)
vec_j_hor = obj_j_hor.vectors()

obj_h_hor = Horizons(id=OBJECT['HEKTOR']['id'], location='@sun', epochs=epochs)
vec_h_hor = obj_h_hor.vectors()

In [ ]:
def get_rotation_coords(vec_target, vec_center):
    theta = np.arctan2(vec_center['y'], vec_center['x'])
    
    x_rot = vec_target['x'] * np.cos(theta) + vec_target['y'] * np.sin(theta) - (vec_center['x'] * np.cos(theta) + vec_center['y'] * np.sin(theta))
    y_rot = -vec_target['x'] * np.sin(theta) + vec_target['y'] * np.cos(theta) - (-vec_center['x'] * np.sin(theta) + vec_center['y'] * np.cos(theta))
    z_rot = vec_target['z'] - vec_center['z']
    
    return pd.dataframe({'x': x_rot, 'y': y_rot, 'z': z_rot})

## ICRF

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(vec_h['x'], vec_h['y'], label=OBJECT['HEKTOR']['name'], color=OBJECT['HEKTOR']['color'], alpha=0.7, zorder=2)
ax.plot(vec_j['x'], vec_j['y'], label=OBJECT['JUPITER']['name'], color=OBJECT['JUPITER']['color'], alpha=0.7, zorder=2)
ax.scatter(0, 0, label=OBJECT['SUN']['name'], color=OBJECT['SUN']['color'], s=40, zorder=2)

ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel('$x$ (au)')
ax.set_ylabel('$y$ (au)')
ax.legend()
ax.grid(True, linestyle=':')

fig.savefig('hektor_icrf.pdf', format='pdf', bbox_inches='tight')
plt.show()

## Jupiter-centered Rotating Frame

In [ ]:
# Rotational Coordinates
vec_h_rot = get_rotation_coords(vec_h, vec_j)
vec_h_rot_hor = get_rotation_coords(vec_h_hor, vec_j_hor)

# L4 Calculation
d_j = np.sqrt(vec_j['x']**2 + vec_j['y']**2 + vec_j['z']**2)
l4_x = -0.5 * d_j
l4_y = (np.sqrt(3) / 2) * d_j
l4_z = 0 * d_j

l4_dist = np.sqrt((vec_h_rot['x'] - l4_x)**2 + (vec_h_rot['y'] - l4_y)**2 + (vec_h_rot['z'] - l4_z)**2)

print(f"Maximum distance from L4 point: {l4_dist.max():.3f} au")

# Plotting
fig = plt.figure(figsize=(6, 6))
gs = fig.add_gridspec(2, 2, 
                      width_ratios=[3, 2],
                      height_ratios=[3, 2])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[0, 1], sharey=ax1)

ax1.plot(vec_h_rot['x'], vec_h_rot['y'], label=f"Simulated Trajectory", color=OBJECT['HEKTOR']['color'], alpha=0.3, zorder=2)
ax1.scatter(vec_h_rot_hor['x'], vec_h_rot_hor['y'], label=f"Horizons (10 years)", color=OBJECT['HEKTOR']['color'], marker='+', s=40, zorder=3)
ax1.plot(l4_x, l4_y, color='darkred', label='Sun--Jupiter $L_4$', zorder=3)
ax1.scatter(0, 0, label=OBJECT['JUPITER']['name'], color=OBJECT['JUPITER']['color'], s=40, zorder=3)

ax1.set_aspect('equal')
ax1.set_ylabel('$y$ (au)')
ax1.set_xlim(-5.5, 0.5)
ax1.set_ylim(-0.5, 5.5)
ax1.grid(True, linestyle=':')

ax2.plot(vec_h_rot['x'], vec_h_rot['z'], color=OBJECT['HEKTOR']['color'], alpha=0.3, zorder=2)
ax2.scatter(vec_h_rot_hor['x'], vec_h_rot_hor['z'], color=OBJECT['HEKTOR']['color'], marker='+', s=40, zorder=3)
ax2.plot(l4_x, l4_z, color='darkred',zorder=3)
ax2.scatter(0, 0, label=OBJECT['JUPITER']['name'], color=OBJECT['JUPITER']['color'], s=40, zorder=3)

ax2.set_aspect('equal')
ax2.set_xlabel('$x$ (au)')
ax2.set_ylabel('$z$ (au)')
ax2.set_ylim(-2, 2)
ax2.grid(True, linestyle=':')

ax3.plot(vec_h_rot['z'], vec_h_rot['y'], color=OBJECT['HEKTOR']['color'], alpha=0.3, zorder=2)
ax3.scatter(vec_h_rot_hor['z'], vec_h_rot_hor['y'], color=OBJECT['HEKTOR']['color'], marker='+', s=40, zorder=3)
ax3.plot(l4_z, l4_y, color='darkred', zorder=3)
ax3.scatter(0, 0, label=OBJECT['JUPITER']['name'], color=OBJECT['JUPITER']['color'], s=40, zorder=3)

ax3.set_aspect('equal')
ax3.set_xlabel('$z$ (au)')
ax3.set_ylim(-0.5, 5.5)
ax3.grid(True, linestyle=':')

ax_legend = fig.add_subplot(gs[1, 1])
ax_legend.axis('off')
handles, labels = ax1.get_legend_handles_labels()
ax_legend.legend(handles, labels, loc='center')

fig.align_ylabels()
fig.savefig('hektor_rot.pdf', format='pdf', bbox_inches='tight')
plt.show()